<a href="https://colab.research.google.com/github/Amulya-Rao-mj/GENAI-lab/blob/main/aichatbot_using_langchain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-groq streamlit pyngrok

In [4]:
import os
from getpass import getpass

GROQ_API_KEY = getpass("Enter your new Groq API key: ")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

print("API key configured successfully!")

Enter your new Groq API key: ··········
API key configured successfully!


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0.7
)

response = llm.invoke("Hello! Introduce yourself.")

print(response.content)

Hi there! 👋 I’m ChatGPT, a conversational AI developed by OpenAI. I’m built on the GPT‑4 architecture and can help answer questions, brainstorm ideas, explain concepts, draft emails, write stories, and much more. I’m here to chat, learn, and assist you with whatever you need—so feel free to ask me anything!


In [6]:

%%writefile app.py

import os
import streamlit as st
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# Page configuration
st.set_page_config(
    page_title="My AI Chatbot",
    page_icon="🤖",
    layout="centered"
)

# Styling
st.markdown("""
<style>
.stApp {
    background-color: #111827;
    color: white;
}
h1 {
    color: #60A5FA;
    text-align: center;
}
</style>
""", unsafe_allow_html=True)

# Session memory
if "messages" not in st.session_state:
    st.session_state.messages = []

# Initialize model
@st.cache_resource
def get_model():
    return ChatGroq(
        model="openai/gpt-oss-20b",
        api_key=os.environ["GROQ_API_KEY"],
        temperature=0.7
    )

llm = get_model()

# Sidebar
with st.sidebar:
    st.title("⚙️ Settings")
    st.write("Model: GPT-OSS 20B")
    st.write("Framework: LangChain")

    if st.button("🗑️ New Chat"):
        st.session_state.messages = []
        st.rerun()

# Main heading
st.title("🤖 AI Chatbot")
st.caption("Your personal AI assistant powered by LangChain")

# Display previous messages
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Chat input
if prompt := st.chat_input("Ask me anything..."):

    st.session_state.messages.append({
        "role": "user",
        "content": prompt
    })

    with st.chat_message("user"):
        st.markdown(prompt)

    # Prepare history
    history = [
        SystemMessage(
            content="You are a helpful and friendly AI assistant."
        )
    ]

    for msg in st.session_state.messages[:-1]:
        if msg["role"] == "user":
            history.append(HumanMessage(content=msg["content"]))
        else:
            history.append(AIMessage(content=msg["content"]))

    history.append(HumanMessage(content=prompt))

    # Generate response
    try:
        with st.chat_message("assistant"):
            placeholder = st.empty()
            answer = ""

            for chunk in llm.stream(history):
                if isinstance(chunk.content, str):
                    answer += chunk.content

                placeholder.markdown(answer + "▌")

            placeholder.markdown(answer)

        st.session_state.messages.append({
            "role": "assistant",
            "content": answer
        })

    except Exception as e:
        st.error(f"Error: {e}")
        st.session_state.messages.pop()


Writing app.py


In [7]:
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true > streamlit.log 2>&1 &

In [8]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Tue, 24 Nov 2026 07:30:14 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7863
last-modified: Tue, 24 Nov 2026 07:26:41 GMT
etag: "15b0870105dce2add8b49397ffbabf4c"
cache-control: no-cache



In [9]:
from google.colab.output import eval_js

url = eval_js("google.colab.kernel.proxyPort(8501)")
print("YOUR CHATBOT:", url)

YOUR CHATBOT: https://8501-m-s-usw1b8-hd60sel6fre6-b.us-west1.prod.r.colab.dev
